# Kadanoff-Baym equations of motion

$$
\left[i \partial_z - h(z) \right]G(z,z^\prime) - \left[\Sigma * G\right](z,z^\prime) = \delta_C(z,z^\prime) \\
\left[-i \partial_z^\prime - h(z^\prime)\right]G(z,z^\prime) - \left[G * \Sigma\right](z,z^\prime)= \delta_C(z,z^\prime)
$$

In [ ]:
import NCA
using AMR, Interpolations

using KadanoffBaym
include("../src/wigner.jl") # Wigner transform
using ElasticArrays

import SpecialFunctions: erf, erfc
import LinearAlgebra: diag

using Plots
default(fmt = :png)

## Initial correlations

### Equilibrium state

In [ ]:
# Solve the problem in equilibrium
model = NCA.SingleImpurityAndersonModel(
    V₀=sqrt(0.2),
    ε₀f=-0.5)

nca = NCA.FreqNCA(
    model=model,
    β=5e2,
    mesh=NCA.LinearMesh(-6.0,6.0,2^16),
    eps=1e-9)

# Don't project cause we haven't solved the problem yet!
imΣ = NCA.solve!(nca; project=false);

In [ ]:
# Calculate Green functions and interpolate them
GᶜR, GᶠR, GᵇR = begin
    
    model.λ₀ = 0.
    Aᶜ = nca.data.A_c
    Aᵇ, Aᶠ = NCA.slave_particle_spkf(nca, imΣ);
    Aᶠ[Aᶠ .< nca.eps] .= 0
    Aᵇ[Aᵇ .< nca.eps] .= 0
    
    # λ₀ shift
    boltz = NCA.bf.(nca.mesh, nca.β)

    # Shift the original mesh
    mesh_shift = nca.mesh.mesh .- Float64(-1/nca.β * log(boltz' * (2*Aᶠ + Aᵇ) * nca.mesh.dx));
    
    Gᶜ = NCA.kramerskronig(nca.mesh, π * Aᶜ, :real) + 1im * π * Aᶜ;
    Gᶠ = NCA.kramerskronig(nca.mesh, π * Aᶠ, :real) + 1im * π * Aᶠ;
    Gᵇ = NCA.kramerskronig(nca.mesh, π * Aᵇ, :real) + 1im * π * Aᵇ;
    
    Gᶜ_interp = LinearInterpolation(mesh_shift,Gᶜ);
    Gᶠ_interp = LinearInterpolation(mesh_shift,Gᶠ);
    Gᵇ_interp = LinearInterpolation(mesh_shift,Gᵇ);
    
    Gᶜ_interp, Gᶠ_interp, Gᵇ_interp
end;

In [ ]:
# Find optimal mesh
mesh, _ = sample_function(x -> GᵇR(x) + GᶠR(x), mesh_shift[1]:0.1:mesh_shift[end]; tol=1e-3);

∫dω(f) = trapz(mesh, f) / (2π)

# Generate grids
boltz = NCA.bf.(mesh, nca.β) .|> Float64; boltz[boltz .== Inf] .= 0.0;
bose = NCA.nb.(mesh, nca.β); bose[bose .== Inf] .= 0.0;
fermi = NCA.nf.(mesh, nca.β);

In [ ]:
# Assert GFs are normalized!
println(∫dω(GᶜR(mesh) - conj(GᶜR(mesh))) |> imag)
println(∫dω(GᶠR(mesh) - conj(GᶠR(mesh))) |> imag)
println(∫dω(GᵇR(mesh) - conj(GᵇR(mesh))) |> imag)

plot(mesh, GᵇR.(mesh) |> imag)
plot!(mesh, GᶠR.(mesh) |> imag)

## Generating a gaussian pulse

Consider a gaussian pulse
$$\xi(t) = \left(\frac{\Omega^2}{2 \pi}\right)^{1/4}\exp{\left(-\frac{\Omega^2}{4}(t-t_c)^2\right)}$$

Consider a modulator cavity given by the triple $(S_M, L_M, H_M)$

for which $L_M = \lambda(t) a$ where $\lambda(t) \triangleq \frac{\xi(t)}{\sqrt{\int_t^\infty \textrm{d}s |\xi(s)|^2}}$

In [ ]:
# Gaussian pulse parameters
const tc = 40.  # peak arrival time
const Ω = 0.12 # frequency bandwidth of the pulse

function ξ(t)
    return (Ω^2 / (2π))^(1/4) * exp(-(t - tc)^2 * (Ω^2 / 4))
end

# Pump and loss terms
const Γ = 2.
λ(t::Float64) = iszero(Γ) ? 0.0 : Γ * ξ(t) / sqrt(0.5 * erfc((t - tc) * Ω / sqrt(2)))
γ(t::Float64) = iszero(Γ) ? 0.0 : Γ * ξ(t)

# Photon field Hamiltonian
h(t::Float64) = 1e-2 - 0.5im * (λ(t) + γ(t))

# Artificial width of the peak
Σᵞ = 1e-2

# Initial field occupation
n₀ = 0.0 

#### Initial conditions

$$
G^<(t_0, t_0) = \mathcal{G}(0, 0^+) = \frac{i}{\beta} \sum_n e^{+i \omega_n 0^+} \mathcal{G}(i \omega_n) = \mp i \int_{-\infty}^{+\infty} \frac{\textrm{d} \omega}{2\pi} f(\omega) A(\omega)
$$
$$
G^>(t_0, t_0) = \mathcal{G}(0^+, 0) = \frac{i}{\beta} \sum_n e^{-i \omega_n 0^+} \mathcal{G}(i \omega_n) = - i \int_{-\infty}^{+\infty} \frac{\textrm{d} \omega}{2\pi} \left(1 \pm f(\omega)\right) A(\omega)
$$

In [ ]:
u0 = begin
    Aᶜ = x -> 1/(π) * imag(GᶜR(x))
    Aᶠ = x -> 1/(π) * imag(GᶠR(x))
    Aᵇ = x -> 1/(π) * imag(GᵇR(x))
    
    G₀ᶜL = +1.0im * ∫dω(fermi .* Aᶜ.(mesh)) * (2π)
    G₀ᶜL = GreenFunction(G₀ᶜL * ones(1,1), Lesser)
    
    G₀ᶜG = -1.0im * (1 - ∫dω(fermi .* Aᶜ.(mesh)) * (2π))
    G₀ᶜG = GreenFunction(G₀ᶜG * ones(1,1), Greater)
    
    G₀ᶠL = +1.0im * ∫dω(boltz .* Aᶠ.(mesh)) * (2π)
    G₀ᶠL = GreenFunction(G₀ᶠL * ones(1,1), Lesser)

    G₀ᶠG = -1.0im * (1 - ∫dω(boltz .* Aᶠ.(mesh)) * (2π))
    G₀ᶠG = GreenFunction(G₀ᶠG * ones(1,1), Greater)

    G₀ᵇL = -1.0im * ∫dω(boltz .* Aᵇ.(mesh)) * (2π)
    G₀ᵇL = GreenFunction(G₀ᵇL * ones(1,1), Lesser)

    G₀ᵇG = -1.0im * (1 + ∫dω(boltz .* Aᵇ.(mesh)) * (2π))
    G₀ᵇG = GreenFunction(G₀ᵇG * ones(1,1), Greater)
    
    G₀ᵞL = -1.0im * n₀
    G₀ᵞL = GreenFunction(G₀ᵞL * ones(1,1), Lesser)
    
    G₀ᵞG = -1.0im * (1 + n₀)
    G₀ᵞG = GreenFunction(G₀ᵞG * ones(1,1), Greater)
    
   [G₀ᶜL, G₀ᶜG, G₀ᶠL, G₀ᶠG, G₀ᵇL, G₀ᵇG, G₀ᵞL, G₀ᵞG];
end

# u0_line = begin
#     [[GᶜR(mesh)], [conj(GᶜR(mesh))], [GᶠR(mesh)], [conj(GᶠR(mesh))], [GᵇR(mesh)], [conj(GᵇR(mesh))]]
# end

# u0 = [u0, u0_line];

### ODEs

$$
i \partial_t G^\gtrless(t,t^\prime) =
h(t) G^\gtrless(t,t^\prime)
+ \int_{t_0}^t \textrm{d}\bar{t} \left[\Sigma^>(t,\bar{t}) - \Sigma^<(t,\bar{t})\right] G^\gtrless(\bar{t},t^\prime)
- \int_{t_0}^{t^\prime} \textrm{d}\bar{t} \Sigma^\gtrless(t, \bar{t}) \left[G^>(\bar{t},t^\prime) - G^<(\bar{t},t^\prime)\right]
- i \int_0^\beta \textrm{d}\tau \Sigma^\rceil(t,\tau) G^\lceil(\tau,t^\prime)
$$
$$
-i \partial_{t^\prime} G^\gtrless(t,t^\prime) =
G^\gtrless(t,t^\prime) h(t)^\dagger
+ \int_{t_0}^{t} \textrm{d}\bar{t} \left[G^>(t,\bar{t}) - G^<(t,\bar{t})\right] \Sigma^\gtrless(\bar{t},t^\prime)
- \int_{t_0}^{t^\prime} \textrm{d}\bar{t} G^\gtrless(t,\bar{t}) \left[\Sigma^>(\bar{t},t^\prime) - \Sigma^<(\bar{t},t^\prime)\right] 
- i \int_0^\beta \textrm{d}\tau G^\rceil(t,\tau)\Sigma^\lceil(\tau,t^\prime) 
$$

In [ ]:
σ = 0.5 # Conduction electron width
Γ₀ = 0.1 # Light-matter coupling
V₀ = model.V₀ # Hybridization strength

# Define commutator
⋊(a,b) = a * b - b * adjoint(a)

"""
Vertical stepping
"""
function f_vert(u, times, t1, t2)
    # Real-time collision integral 
    ∫dt(i,j,A::Function,B::GreenFunction) = sign(j-i) * trapz(times[min(i,j):max(i,j)], [A(t1,t) * B[t,t2] for t=min(i,j):max(i,j)])

    # Maps a G˥ to a G⌐
    ⌐(A) = conj(reverse(A))
    
    # unpack Green functions
    GᶜL, GᶜG, GᶠL, GᶠG, GᵇL, GᵇG, GᵞL, GᵞG = u#[1]
#     Gᶜ˥p, Gᶜ˥m, Gᶠ˥p, Gᶠ˥m, Gᵇ˥p, Gᵇ˥m = u[2]
    
    # conduction electron
    dGᶜG = -1.0im * (- 1.0im * σ * GᶜG[t1,t2])
    dGᶜL = -1.0im * (- 1.0im * σ * GᶜL[t1,t2])
    
    # pseudo fermion
    ΣᶠL(i,j) = GᶜL[i,j] * (V₀^2 + Γ₀^2 * (GᵞL[i,j] + GᵞG[j,i])) * GᵇL[i,j]
    ΣᶠG(i,j) = GᶜG[i,j] * (V₀^2 + Γ₀^2 * (GᵞL[i,j] + GᵞG[j,i])) * GᵇG[i,j]
#     Σᶠ˥p = V₀^2 * Gᶜ˥p[t1] .* Gᵇ˥p[t1]
#     Σᶠ˥m = V₀^2 * Gᶜ˥m[t1] .* Gᵇ˥m[t1]
    
    dGᶠG = -1.0im * (model.ε₀f * GᶠG[t1,t2] + ∫dt(t1,t2,ΣᶠG,GᶠG))
    dGᶠL = -1.0im * (model.ε₀f * GᶠL[t1,t2] + ∫dt(1,t1,ΣᶠG,GᶠL) - ∫dt(1,t2,ΣᶠG,GᶠG))
#         .- ∫dω((1 .- fermi) .* (Σᶠ˥p .* ⌐(Gᶠ˥m[t2]) - Σᶠ˥m .* ⌐(Gᶠ˥p[t2]))))
    
    # slave boson
    ΣᵇL(i,j) = V₀^2 * GᶠL[i,j] * GᶜG[j,i] + Γ₀^2 * (GᵞL[i,j] + GᵞG[j,i]) * GᶠL[i,j] * GᶜG[j,i] 
    ΣᵇG(i,j) = V₀^2 * GᶠG[i,j] * GᶜL[j,i] + Γ₀^2 * (GᵞL[i,j] + GᵞG[j,i]) * GᶠG[i,j] * GᶜL[j,i] 
#     Σᵇ˥p = V₀^2 * Gᶠ˥p[t1] .* ⌐(Gᶜ˥p[t1])
#     Σᵇ˥m = V₀^2 * Gᶠ˥m[t1] .* ⌐(Gᶜ˥m[t1])
    
    dGᵇG = -1.0im * (0.0 * GᵇG[t1,t2] + ∫dt(t1,t2,ΣᵇG,GᵇG))
    dGᵇL = -1.0im * (0.0 * GᵇL[t1,t2] + ∫dt(1,t1,ΣᵇG,GᵇL) - ∫dt(1,t2,ΣᵇG,GᵇG))
#         .- ∫dω((1 .+ bose) .* (Σᵇ˥p .* ⌐(Gᵇ˥m[t2]) - Σᵇ˥m .* ⌐(Gᵇ˥p[t2]))))
    
    # thz pulse
    dGᵞL = -1.0im * ((h(times[t1]) + 1.0im * γ(times[t1]) - 1.0im * Σᵞ) * GᵞL[t1,t2])
    dGᵞG = -1.0im * ((h(times[t1]) + 1.0im * γ(times[t1]) - 1.0im * Σᵞ) * GᵞG[t1,t2])
    
    return [dGᶜL, dGᶜG, dGᶠL, dGᶠG, dGᵇL, dGᵇG, dGᵞL, dGᵞG]
end

function f_diag(u, times, t1, t2)
    @assert t1 == t2
    println(times[t1])
    
    # Real-time collision integral 
    ∫dt(i,j,A::Function,B::GreenFunction) = sign(j-i) * trapz(times[min(i,j):max(i,j)], [A(t1,t) * B[t,t2] for t=min(i,j):max(i,j)])
    ∫dt(i,j,A::GreenFunction,B::Function) = sign(j-i) * trapz(times[min(i,j):max(i,j)], [A[t1,t] * B(t,t2) for t=min(i,j):max(i,j)])
    
    # Maps a G˥ to a G⌐
    ⌐(A) = conj(reverse(A))
    
    # unpack Green functions
    GᶜL, GᶜG, GᶠL, GᶠG, GᵇL, GᵇG, GᵞL, GᵞG = u#[1]
#     Gᶜ˥p, Gᶜ˥m, Gᶠ˥p, Gᶠ˥m, Gᵇ˥p, Gᵇ˥m = u[2]
    
    # conduction electron
    dGᶜG = -1.0im * zero(GᶜG[t1,t2])
    dGᶜL = -1.0im * zero(GᶜL[t1,t2])
    
    # pseudo fermion
    ΣᶠL(i,j) = V₀^2 * GᶜL[i,j] * GᵇL[i,j] + Γ₀^2 * GᶜL[i,j] * (GᵞL[i,j] + GᵞG[j,i]) * GᵇL[i,j]
    ΣᶠG(i,j) = V₀^2 * GᶜG[i,j] * GᵇG[i,j] + Γ₀^2 * GᶜG[i,j] * (GᵞL[i,j] + GᵞG[j,i]) * GᵇG[i,j]
#     Σᶠ˥p = V₀^2 * Gᶜ˥p[t1] .* Gᵇ˥p[t1]
#     Σᶠ˥m = V₀^2 * Gᶜ˥m[t1] .* Gᵇ˥m[t1]
    
    dGᶠG = -1.0im * (zero(GᶠG[t1,t2]) + ∫dt(t1,t2,ΣᶠG,GᶠG) - ∫dt(t1,t2,GᶠG,ΣᶠG))
    dGᶠL = -1.0im * (zero(GᶠL[t1,t2]) + ∫dt(1,t1,ΣᶠG,GᶠL) - ∫dt(1,t2,ΣᶠG,GᶠG))
#         .- ∫dω((1 .- fermi) .* (Σᶠ˥p .* ⌐(Gᶠ˥m[t1]) - Σᶠ˥m .* ⌐(Gᶠ˥p[t1])))
#         .+ ∫dω((1 .- fermi) .* (Gᶠ˥m[t1] .* ⌐(Σᶠ˥p) - Gᶠ˥p[t1] .* ⌐(Σᶠ˥m))))
    
    # slave boson
    ΣᵇL(i,j) = V₀^2 * GᶠL[i,j] * GᶜG[j,i] + Γ₀^2 * (GᵞL[i,j] + GᵞG[j,i]) * GᶠL[i,j] * GᶜG[j,i] 
    ΣᵇG(i,j) = V₀^2 * GᶠG[i,j] * GᶜL[j,i] + Γ₀^2 * (GᵞL[i,j] + GᵞG[j,i]) * GᶠG[i,j] * GᶜL[j,i] 
#     Σᵇ˥p = V₀^2 * Gᶠ˥p[t1] .* ⌐(Gᶜ˥p[t1])
#     Σᵇ˥m = V₀^2 * Gᶠ˥m[t1] .* ⌐(Gᶜ˥m[t1])
    
    dGᵇG = -1.0im * (zero(GᵇG[t1,t2]) + ∫dt(t1,t2,ΣᵇG,GᵇG) - ∫dt(t1,t2,GᵇG,ΣᵇG))
    dGᵇL = -1.0im * (zero(GᵇL[t1,t2]) + ∫dt(1,t1,ΣᵇG,GᵇL) - ∫dt(1,t2,ΣᵇG,GᵇG))
#         .- ∫dω((1 .+ bose) .* (Σᵇ˥p .* ⌐(Gᵇ˥m[t1]) - Σᵇ˥m .* ⌐(Gᵇ˥p[t1])))    
#         .+ ∫dω((1 .+ bose) .* (Gᵇ˥m[t1] .* ⌐(Σᵇ˥p) - Gᵇ˥p[t1] .* ⌐(Σᵇ˥m))))
    
    # thz pulse
    dGᵞL = -1.0im * (h(times[t1]) ⋊ GᵞL[t1,t2] + 1.0im * γ(times[t1]) * (GᵞL[t1,t2] + GᵞG[t1,t2]))
    dGᵞG = -1.0im * (h(times[t1]) ⋊ GᵞG[t1,t2] + 1.0im * λ(times[t1]) * (GᵞG[t1,t2] - GᵞL[t1,t2]) 
        + 2.0im * γ(times[t1]) * GᵞG[t1,t2])
    
    return [dGᶜL, dGᶜG, dGᶠL, dGᶠG, dGᵇL, dGᵇG, dGᵞL, dGᵞG]
end

# function f_line(u, times, t1)
#     # Integral 
#     ∫dt(A::Function,B) = trapz(times, [A(t1,t) * B[t] for t=1:t1])
    
#     # Maps a G˥ to a G⌐
#     ⌐(A) = conj(reverse(A))
    
#     # unpack Green functions
#     GᶜL, GᶜG, GᶠL, GᶠG, GᵇL, GᵇG, = u[1]
#     Gᶜ˥p, Gᶜ˥m, Gᶠ˥p, Gᶠ˥m, Gᵇ˥p, Gᵇ˥m = u[2]

#     # f-electron self-energies
#     ΣᶠL(i,j) = V₀^2 * GᶜL[i,j] * GᵇL[i,j] + Γ₀^2 * GᶜL[i,j] * (GᵞL[i,j] + GᵞG[j,i]) * GᵇL[i,j]
#     ΣᶠG(i,j) = V₀^2 * GᶜG[i,j] * GᵇG[i,j] + Γ₀^2 * GᶜG[i,j] * (GᵞL[i,j] + GᵞG[j,i]) * GᵇG[i,j]
#     Σᶠ˥p = V₀^2 * Gᶜ˥p[t1] .* Gᵇ˥p[t1]
#     Σᶠ˥m = V₀^2 * Gᶜ˥m[t1] .* Gᵇ˥m[t1]
    
#     # slave boson self-energies
#     ΣᵇL(i,j) = V₀^2 * GᶠL[i,j] * GᶜG[j,i] + Γ₀^2 * (GᵞL[i,j] + GᵞG[j,i]) * GᶠL[i,j] * GᶜG[j,i] 
#     ΣᵇG(i,j) = V₀^2 * GᶠG[i,j] * GᶜL[j,i] + Γ₀^2 * (GᵞL[i,j] + GᵞG[j,i]) * GᶠG[i,j] * GᶜL[j,i] 
#     Σᵇ˥p = V₀^2 * Gᶠ˥p[t1] .* ⌐(Gᶜ˥p[t1])
#     Σᵇ˥m = V₀^2 * Gᶠ˥m[t1] .* ⌐(Gᶜ˥m[t1])

#     dGᶜ˥p = -1.0im * (-1.0im * σ * Gᶜ˥p[t1]) #+ ∫dt(ζ ΣᶜG, Gᶜ˥p) - ∫dt(ζ ΣᶜL, Gᶜ˥p) + ζ Σᶜ˥p(t,:) .* GᶜR)
#     dGᶜ˥m = -1.0im * (-1.0im * σ * Gᶜ˥m[t1]) #+ ∫dt(ζ ΣᶜG, Gᶜ˥m) - ∫dt(ζ ΣᶜL, Gᶜ˥m) + ζ Σᶜ˥m(t,:) .* adjoint(GᶜR))
    
#     dGᶠ˥p = -1.0im * (+model.ε₀f * Gᶠ˥p[t1] + ∫dt(ΣᶠG, Gᶠ˥p) + Σᶠ˥p .* GᶠR) # - ∫dt(ζ ΣᶠL, Gᶠ˥p)
#     dGᶠ˥m = -1.0im * (+model.ε₀f * Gᶠ˥m[t1] + ∫dt(ΣᶠG, Gᶠ˥p) + Σᶠ˥m .* conj(GᶠR)) # - ∫dt(ζ ΣᶠL, Gᶠ˥m)

#     dGᵇ˥p = -1.0im * (+0.0 * zero(Gᵇ˥p[t1]) + ∫dt(ΣᵇG, Gᶜ˥p) + Σᵇ˥p .* GᵇR) # - ∫dt(ζ ΣᵇL, Gᵇ˥p)
#     dGᵇ˥m = -1.0im * (+0.0 * zero(Gᵇ˥m[t1]) + ∫dt(ΣᵇG, Gᵇ˥p) + Σᵇ˥m .* conj(GᵇR)) # - ∫dt(ζ ΣᵇL, Gᵇ˥m)

#     return [dGᶜ˥p, dGᶜ˥m, dGᶠ˥p, dGᶠ˥m, dGᵇ˥p, dGᵇ˥m]
# end

In [ ]:
# ODE problem is defined by the rhs, initial condition and time span
tspan = (0.0,150.)

u, times = kbsolve(f_vert, f_diag, u0, tspan[1], tspan[2];
    dt=1e-3, max_dt=0.25,adaptive=true, atol=1e-6, rtol=1e-4)

In [ ]:
GᶜL = u[1]
GᶜG = u[2]

GᶠL = u[3]
GᶠG = u[4]

GᵇL = u[5]
GᵇG = u[6]

GᵞL = u[7];
GᵞG = u[8];

In [ ]:
plot(times, GᵞL.data |> diag .|> imag .|> -, label="Lesser")
plot!(times, GᵞG.data |> diag .|> imag .|> -, label="Greater")
plot!(twinx(), times, ξ.(times), color=:green, legend=:right, label="Pulse", line=:dot, xlabel=L"t")

In [ ]:
heatmap(times, times, -imag(GᶠL.data))

In [ ]:
heatmap(times, times, -imag(GᵇL.data))

In [ ]:
heatmap(times, times, -imag(GᵞL.data))

In [ ]:
ρ = wigner_transform(GᵇG.data - GᵇL.data; Nf=4096*2);
wigner_plot!(real(ρ), times)#; ylims=(-10,10))

In [ ]:
ρ = wigner_transform(GᶠG.data - GᶠL.data; Nf=4096*2);
wigner_plot!(real(ρ), times)#; ylims=(-10,10))

In [ ]:
ρ = wigner_transform(GᵞG.data - GᵞL.data; Nf=4096*2);
wigner_plot!(real(ρ), times; ylims=(-3,3))